<a href="https://colab.research.google.com/github/setugujar2-source/deepLearning-Projects/blob/main/ResNet_CIFAR10_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install timm transformers torchinfo

In [2]:
!ls -lh my_dataset.zip

ls: cannot access 'my_dataset.zip': No such file or directory


In [3]:
import torch
import torchvision
import timm
import transformers

print("PyTorch 版本:", torch.__version__)
print("Torchvision 版本:", torchvision.__version__)
print("Timm 版本:", timm.__version__)
print("Transformers 版本:", transformers.__version__)
print("CUDA 是否可用:", torch.cuda.is_available())

PyTorch 版本: 2.11.0+cu128
Torchvision 版本: 0.26.0+cu128
Timm 版本: 1.0.29
Transformers 版本: 5.16.1
CUDA 是否可用: True


In [4]:
# 1. 安装 7-Zip 工具
!apt-get install p7zip-full -y

# 2. 使用 7-Zip 强制解压
!7z x my_dataset.zip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+transitional.1).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan         
ERROR: errno=2 : No such file or directory
my_dataset.zip



System ERROR:
errno=2 : No such file or directory


In [5]:
!ls my_dataset/train

ls: cannot access 'my_dataset/train': No such file or directory


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

# 1. 确认设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'当前使用设备: {device}')

# 2. 数据预处理（ResNet 要求输入 224x224，并使用 ImageNet 的均值方差）
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. 加载数据集（请确保你已把 my_dataset 上传到了 Colab 左侧的文件夹里）
# 如果你还没上传，可以新建一个带数据集的代码块，或者在本地跑
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
# 简单划分 20% 作为验证集
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(123))
val_dataset.dataset.transform = val_test_transforms  # 验证集不进行增强

test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# 4. 加载预训练 ResNet18 并修改输出层
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10) # 修改为你的三分类
model = model.to(device)

# 5. 定义损失函数和优化器（学习率调小）
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-5) # 微调学习率要很小

# 6. 训练循环（简写版，直接跑）
epochs = 5 # 预训练模型收敛极快，5轮足够
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # 验证
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}/{epochs} | 训练准确率: {correct/total:.4f} | 验证准确率: {val_correct/val_total:.4f}")

# 7. 测试集评估
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
print(f"\n测试集准确率: {test_correct/test_total:.4f}")

当前使用设备: cuda


100%|██████████| 170M/170M [40:06<00:00, 70.8kB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s]


Epoch 1/5 | 训练准确率: 0.8550 | 验证准确率: 0.9392
Epoch 2/5 | 训练准确率: 0.9536 | 验证准确率: 0.9464


In [ ]:
!ls

In [ ]:
!ls my_dataset